# TideLock feasibility lab

A small, reproducible experiment before building a dashboard. **All results are synthetic; this is not a field-safe controller.**

Use the copy in My Drive/TideLock. Run each cell in order. This notebook does not clone over or overwrite your project. It writes new evidence to results/colab.


In [ ]:
from pathlib import Path
import os, sys, subprocess
try:
    from google.colab import drive
except ImportError:
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT = next((p for p in candidates if (p / "tidelock").is_dir()), None)
    if PROJECT is None:
        raise RuntimeError("Open this notebook from the TideLock project or notebooks directory")
else:
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/TideLock")
if not (PROJECT / "tidelock").is_dir():
    raise FileNotFoundError("Expected My Drive/TideLock/tidelock; check your project folder")
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print("Project:", PROJECT)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)


## Reproduce the compact demonstration

Every controller uses the same inputs and initial conditions in each case. The sensor-bias case is a separate paired scenario, not a fault spliced into one favored run. The extreme-tide case is expected to exceed the safety limit.


In [ ]:
from tidelock.benchmark import run
OUTPUT = PROJECT / "results" / "colab"
run(OUTPUT, seeds=[11], scenarios=["ordinary", "compound", "sensor_bias", "extreme_tide"])


In [ ]:
from IPython.display import display, Image, Markdown
display(Markdown((OUTPUT / "RESULTS.md").read_text().split("![Compound")[0]))
for name in ["compound", "sensor_bias", "extreme_tide", "pareto"]:
    display(Image(filename=str(OUTPUT / f"{name}.png")))


## Change one scenario and compare fairly

Available cases are listed below. Never compare controllers using different seeds or fault times. New outputs preserve all three strategies.


In [ ]:
from tidelock.scenarios import SCENARIOS
print(SCENARIOS)
SCENARIO = "near_full"
SEED = 23
run(PROJECT / "results" / "custom", seeds=[SEED], scenarios=[SCENARIO])


## Full pilot

Uncomment the next line to repeat all 255 runs. Timing depends on the machine; the notebook is not a real-time control system. Colab versions may differ from requirements-lock.txt, so check the generated manifest.


In [ ]:
# run(PROJECT / "results" / "colab_full", seeds=[11, 23, 37, 51, 79], scenarios=SCENARIOS)


## Interpretation

Ordinary and high-tide-only cases show no predictive benefit in the saved pilot. Sensor faults and near-full baskets can exceed the safety limit. Compare capture, escaped mass **and suspended inventory**, and read docs/prior_art.md before describing novelty. This is a first-stage feasibility study, not an accepted software-only final submission or validated flood-protection design.
